# Simple Shakespeare causal transformer

A minimal character-level causal language model in PyTorch, trained on the normalized Shakespeare dataset.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import math
import sys

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

repo_root = Path.cwd()
if not (repo_root / "modelling").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

from modelling.shakespeare import build_datasets

In [ ]:
@dataclass
class Config:
    block_size: int = 64
    batch_size: int = 64
    n_embd: int = 64
    n_head: int = 4
    n_layer: int = 2
    dropout: float = 0.1
    learning_rate: float = 3e-4
    train_fraction: float = 0.9
    num_steps: int = 2000
    eval_interval: int = 200
    eval_batches: int = 20


config = Config()
device = torch.device("mps")

train_dataset, test_dataset, vocab = build_datasets(
    block_size=config.block_size,
    train_fraction=config.train_fraction,
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, drop_last=True)
train_iter = iter(train_loader)

print(f"device: {device}")
print(f"vocab size: {vocab.size}")
print(f"train batches: {len(train_loader)}")
print(f"test batches: {len(test_loader)}")

device: mps
vocab size: 27
train batches: 14816
test batches: 1645


In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float) -> None:
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")

        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, channels = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        y = self.proj(y)
        return self.resid_dropout(y)


class FeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, cfg: Config) -> None:
        super().__init__()
        self.block_size = cfg.block_size
        self.token_embedding = nn.Embedding(vocab_size, cfg.n_embd)
        self.position_embedding = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout) for _ in range(cfg.n_layer)]
        )
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, vocab_size)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch_size, seq_len = idx.shape
        if seq_len > self.block_size:
            raise ValueError("sequence length exceeds block size")

        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            next_token_logits = logits[:, -1, :]
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx


model = CausalTransformerLM(vocab.size, config).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_params:,}")

parameters: 107,675


In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)


def next_train_batch() -> tuple[torch.Tensor, torch.Tensor]:
    global train_iter
    try:
        xb, yb = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        xb, yb = next(train_iter)
    return xb.to(device), yb.to(device)


@torch.no_grad()
def estimate_loss() -> dict[str, float]:
    model.eval()
    losses = {}
    for split_name, loader in (("train", train_loader), ("test", test_loader)):
        split_losses = []
        loader_iter = iter(loader)
        for _ in range(config.eval_batches):
            try:
                xb, yb = next(loader_iter)
            except StopIteration:
                break
            xb = xb.to(device)
            yb = yb.to(device)
            _, loss = model(xb, yb)
            split_losses.append(loss.item())
        losses[split_name] = sum(split_losses) / len(split_losses)
    model.train()
    return losses

In [5]:
model.train()
for step in range(config.num_steps):
    xb, yb = next_train_batch()
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % config.eval_interval == 0 or step == config.num_steps - 1:
        losses = estimate_loss()
        print(
            f"step {step:4d} | train loss {losses['train']:.4f} | test loss {losses['test']:.4f}"
        )

step    0 | train loss 3.4105 | test loss 3.4047
step  200 | train loss 2.4103 | test loss 2.4375
step  400 | train loss 2.3397 | test loss 2.3839
step  600 | train loss 2.2998 | test loss 2.3581
step  800 | train loss 2.2667 | test loss 2.3276
step 1000 | train loss 2.2331 | test loss 2.2957
step 1200 | train loss 2.1954 | test loss 2.2546
step 1400 | train loss 2.1351 | test loss 2.2032
step 1600 | train loss 2.0888 | test loss 2.1562
step 1800 | train loss 2.0514 | test loss 2.1073
step 1999 | train loss 2.0065 | test loss 2.0775


In [6]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300)[0].tolist()
print(vocab.decode(generated))

 onre ourming shan word buld prin ward wiserts old red gat med thoune ed thoss thaso promblome shepot sone fromulmes waxferis dind inares antned us gof bus ablecest this unge ars beathoond lons merot then zode rexp whes uthe fio of hand brow ksol reers nobry fay huchamerd at and that your yorch no i 
